# Generate Model Comparison LaTeX Table

This notebook reads model metadata from `experiments/QNN_integration/.results-local`, extracts total trainable parameters and quantum-layer trainable parameters, and writes a LaTeX table using the requested Table 2 layout.

In [1]:
from __future__ import annotations

import json
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = print


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "experiments" / "QNN_integration" / ".results-local").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate experiments/QNN_integration/.results-local from the current working directory."
    )


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "experiments" / "QNN_integration" / ".results-local"
ANALYSIS_DIR = PROJECT_ROOT / "experiments" / "QNN_integration" / "analysis"
OUTPUT_TEX = ANALYSIS_DIR / "latex_model_comparison_table.tex"

PIPELINE_ROWS = [
    ("Baseline A", "Baseline A"),
    ("Baseline B", "Baseline B"),
    ("Baseline C", "Baseline C"),
    ("Baseline D", "Baseline D"),
    ("Experiment A", "GEQIE FRQI"),
    ("Experiment B", "GEQIE NEQR"),
]

QUANTUM_LAYER_NAMES = ("VQCLayer", "TorchConnector")

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Output tex:   {OUTPUT_TEX}")

Project root: c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie
Results dir:  c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie\experiments\QNN_integration\.results-local
Output tex:   c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie\experiments\QNN_integration\analysis\latex_model_comparison_table.tex


In [2]:
def parse_int(value: str) -> int:
    return int(value.replace(",", ""))


def format_int(value: int | str | None) -> str:
    if value is None:
        return "--"
    if isinstance(value, int):
        return f"{value:,}"
    return value


def parse_summary_value(summary: str, label: str) -> int:
    match = re.search(rf"{re.escape(label)}:\s*([0-9][0-9,]*)", summary)
    if not match:
        raise ValueError(f"Could not find '{label}' in torchinfo summary.")
    return parse_int(match.group(1))


def parse_quantum_layer_params(summary: str) -> tuple[int, list[tuple[str, int]]]:
    matches = []
    for line in summary.splitlines():
        layer_name = next((name for name in QUANTUM_LAYER_NAMES if name in line), None)
        if layer_name is None:
            continue
        param_match = re.search(r"([0-9][0-9,]*)\s*$", line)
        if param_match:
            matches.append((layer_name, parse_int(param_match.group(1))))
    return sum(value for _, value in matches), matches


def run_sort_key(path: Path) -> tuple[datetime, float]:
    try:
        timestamp = datetime.strptime(path.name, "%d-%m-%Y-%H-%M")
    except ValueError:
        timestamp = datetime.fromtimestamp(0)
    return timestamp, path.stat().st_mtime


def latest_run_dir(pipeline_dir: Path) -> Path:
    run_dirs = [
        path
        for path in pipeline_dir.iterdir()
        if path.is_dir() and any(path.glob("subset_*_best_model_meta.json"))
    ]
    if not run_dirs:
        raise FileNotFoundError(f"No run folders with model metadata found in {pipeline_dir}")
    return max(run_dirs, key=run_sort_key)


def most_common_or_join(values: list[int]) -> int | str | None:
    if not values:
        return None
    unique_values = sorted(set(values))
    if len(unique_values) == 1:
        return unique_values[0]
    counts = Counter(values)
    most_common_count = counts.most_common(1)[0][1]
    tied = sorted(value for value, count in counts.items() if count == most_common_count)
    if len(tied) == 1:
        return tied[0]
    return " / ".join(format_int(value) for value in unique_values)


def latex_escape(text: str) -> str:
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in text)


def load_pipeline_row(display_label: str, folder_name: str) -> dict[str, object]:
    pipeline_dir = RESULTS_DIR / folder_name
    run_dir = latest_run_dir(pipeline_dir)
    meta_paths = sorted(run_dir.glob("subset_*_best_model_meta.json"))

    total_trainable_values = []
    quantum_values = []
    classifier_names = []
    quantum_matches = []

    for meta_path in meta_paths:
        metadata = json.loads(meta_path.read_text(encoding="utf-8"))
        summary = metadata.get("torchinfo_summary") or metadata.get("report_context", {}).get(
            "torchinfo_summary", ""
        )
        context = metadata.get("report_context", {})
        classifier_names.append(context.get("classifier_name") or metadata.get("model", {}).get("class_name"))
        total_trainable_values.append(parse_summary_value(summary, "Trainable params"))
        quantum_param_count, matches = parse_quantum_layer_params(summary)
        quantum_values.append(quantum_param_count)
        quantum_matches.extend(matches)

    classifier_name = next((name for name in classifier_names if name), folder_name)
    return {
        "model_name": f"{display_label}: {classifier_name}",
        "pipeline_folder": folder_name,
        "run_folder": run_dir.name,
        "subset_count": len(meta_paths),
        "total_trainable_parameters": most_common_or_join(total_trainable_values),
        "quantum_layer_trainable_parameters": most_common_or_join(quantum_values),
        "quantum_layers_found": sorted(set(name for name, _ in quantum_matches)),
    }


rows = [load_pipeline_row(display_label, folder_name) for display_label, folder_name in PIPELINE_ROWS]
rows

[{'model_name': 'Baseline A: DenseBaseline',
  'pipeline_folder': 'Baseline A',
  'run_folder': '05-05-2026-10-40',
  'subset_count': 5,
  'total_trainable_parameters': 2570,
  'quantum_layer_trainable_parameters': 0,
  'quantum_layers_found': []},
 {'model_name': 'Baseline B: SimpleCNNBaseline',
  'pipeline_folder': 'Baseline B',
  'run_folder': '05-05-2026-10-41',
  'subset_count': 5,
  'total_trainable_parameters': 38282,
  'quantum_layer_trainable_parameters': 0,
  'quantum_layers_found': []},
 {'model_name': 'Baseline C: PCA + ZZFeatureMap + QNN',
  'pipeline_folder': 'Baseline C',
  'run_folder': '05-05-2026-22-01',
  'subset_count': 5,
  'total_trainable_parameters': 41042,
  'quantum_layer_trainable_parameters': 36,
  'quantum_layers_found': ['TorchConnector']},
 {'model_name': 'Baseline D: CNN + ZZFeatureMap + QNN',
  'pipeline_folder': 'Baseline D',
  'run_folder': '05-05-2026-23-47',
  'subset_count': 5,
  'total_trainable_parameters': 51998,
  'quantum_layer_trainable_param

In [3]:
def make_latex_table(rows: list[dict[str, object]]) -> str:
    lines = [
        "% TABLE 2: Models comparison and their trainable parameters.",
        r"\begin{table}[h]",
        r"\caption{Comparison of model trainable parameters}\label{tab2}%",
        r"\begin{tabular}{@{}lll@{}}",
        r"\toprule",
        "Model name  & Total trainable parameters & Quantum layer trainable parameters \\\\",
        r"\midrule",
    ]

    for row in rows:
        model_name = latex_escape(str(row["model_name"]))
        total = format_int(row["total_trainable_parameters"])
        quantum = format_int(row["quantum_layer_trainable_parameters"])
        lines.append(f"{model_name} & {total} & {quantum} \\\\")

    lines.extend([
        r"\botrule",
        r"\end{tabular}",
        r"\end{table}",
    ])
    return "\n".join(lines)


latex_table = make_latex_table(rows)
OUTPUT_TEX.write_text(latex_table + "\n", encoding="utf-8")

if Markdown is not None:
    display(Markdown("```latex\n" + latex_table + "\n```"))
else:
    print(latex_table)

print(f"Wrote LaTeX table to {OUTPUT_TEX}")

```latex
% TABLE 2: Models comparison and their trainable parameters.
\begin{table}[h]
\caption{Comparison of model trainable parameters}\label{tab2}%
\begin{tabular}{@{}lll@{}}
\toprule
Model name  & Total trainable parameters & Quantum layer trainable parameters \\
\midrule
Baseline A: DenseBaseline & 2,570 & 0 \\
Baseline B: SimpleCNNBaseline & 38,282 & 0 \\
Baseline C: PCA + ZZFeatureMap + QNN & 41,042 & 36 \\
Baseline D: CNN + ZZFeatureMap + QNN & 51,998 & 36 \\
Experiment A: FRQI + QNN + Dense & 5,265 & 135 \\
Experiment B: NEQR + QNN + Dense & 5,265 & 135 \\
\botrule
\end{tabular}
\end{table}
```

Wrote LaTeX table to c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie\experiments\QNN_integration\analysis\latex_model_comparison_table.tex
